# Лабораторная работа №1

## Компьютерная лексикография, терминологические базы (TBX) и Translation Memory (TMX) в Smartcat

**ФИО студента:** Петрова Ульяна Павловна
**Группа:** ИНФА-221
**Вариант:** 10 (предметная область: Автомобилестроение и электротранспорт)

---

**Цель:** научиться программно обрабатывать стандарты обмена терминологическими
данными (TBX) и памятью переводов (TMX), реализовать поиск нечётких совпадений
(Fuzzy Match) и подстановку терминов глоссария, экспортировать результат в
CAT-систему Smartcat.

**Порядок выполнения:**

1. Выполните ячейки Блока 1 (установка библиотек) и Блока 2 (данные).
2. Реализуйте четыре функции в Блоке 4 вместо маркеров `# TODO`.
3. Запустите Блок 5 — все `assert` должны пройти без ошибок.
4. Соберите файлы своего варианта, экспортируйте CSV, оформите отчёт.

**Оценивание (10 баллов):** парсинг TBX — 2, парсинг TMX — 2, Fuzzy Match — 3,
интеграция глоссария — 2, оформление — 1.

## Блок 1. Setup — установка библиотек

In [39]:
!pip install -q Levenshtein pandas lxml

In [40]:
from __future__ import annotations

import os
import re
import unicodedata
from typing import Any, Dict, List, Optional, Sequence, Tuple

import pandas as pd

try:
    from lxml import etree as ET

    LXML_AVAILABLE = True
except ImportError:
    import xml.etree.ElementTree as ET

    LXML_AVAILABLE = False

try:
    import Levenshtein as _lev

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна между двумя строками."""
        return _lev.distance(a, b)

except ImportError:

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна (резервная реализация на чистом Python)."""
        if a == b:
            return 0
        if not a:
            return len(b)
        if not b:
            return len(a)
        previous = list(range(len(b) + 1))
        for i, char_a in enumerate(a, start=1):
            current = [i]
            for j, char_b in enumerate(b, start=1):
                current.append(min(previous[j] + 1,
                                   current[j - 1] + 1,
                                   previous[j - 1] + (char_a != char_b)))
            previous = current
        return previous[-1]


# ВАЖНО: атрибут xml:lang в парсере раскрывается в полное имя с пространством имён
XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"

pd.set_option("display.max_colwidth", 60)
print("Окружение готово. XML backend:", "lxml" if LXML_AVAILABLE else "ElementTree")

Окружение готово. XML backend: lxml


## Блок 2. Теория: структура стандартов

### 2.1. TBX (TermBase eXchange, ISO 30042)

Формат обмена терминологическими базами. Организован **по понятиям**
(concept-oriented): одна `termEntry` = одно понятие, внутри — по одной `langSet`
на каждый язык.

```xml
<martif type="TBX" xml:lang="en">
  <martifHeader>…</martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Предметная область</descrip>
      <langSet xml:lang="en">
        <tig>
          <term>flight controller</term>
          <termNote type="partOfSpeech">noun</termNote>
        </tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp>
          <term>полётный контроллер</term>
        </termGrp></ntig>
      </langSet>
    </termEntry>
  </body></text>
</martif>
```

| Тег | Назначение |
|-----|------------|
| `martif` | корень документа TBX-Core v2 |
| `termEntry` | терминологическая статья (одно понятие) |
| `langSet` | языковая секция, язык задаётся атрибутом `xml:lang` |
| `tig` / `ntig` | компактная / расширенная группа термина |
| `term` | сам термин |
| `termNote` | грамматическая или иная помета (`partOfSpeech`, `usage`) |
| `descrip` | описание: `definition`, `subjectField`, `context` |

### 2.2. TMX (Translation Memory eXchange, версия 1.4b)

Формат обмена памятью переводов. Организован **по сегментам**.

```xml
<tmx version="1.4">
  <header srclang="en" segtype="sentence" datatype="plaintext"
          creationtool="…" adminlang="en" o-tmf="TMX"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Aerospace</prop>
      <tuv xml:lang="en"><seg>The autopilot holds the altitude.</seg></tuv>
      <tuv xml:lang="ru"><seg>Автопилот выдерживает высоту.</seg></tuv>
    </tu>
  </body>
</tmx>
```

| Тег | Назначение |
|-----|------------|
| `header` | метаданные памяти: инструмент, язык оригинала, тип сегментации |
| `tu` | translation unit — единица перевода |
| `tuv` | translation unit variant — вариант на одном языке (`xml:lang`) |
| `seg` | текст сегмента |
| `prop` | произвольное свойство (`domain`, `client`, `project`) |

### 2.3. Типы совпадений в CAT-системах

| Совпадение | Диапазон | Действие переводчика |
|------------|----------|----------------------|
| ICE (101 %) | 100 % + совпал контекст | подстановка без правки |
| Exact | 100 % | проверить контекст |
| High Fuzzy | 95–99 % | косметическая правка |
| Fuzzy | 75–94 % | редактирование |
| No Match | < 75 % | перевод с нуля |

## Блок 3. Данные

Ячейка ниже создаёт **демонстрационные** файлы `demo.tbx` и `demo.tmx` — они
нужны только для автотестов из Блока 5. Файлы своего варианта
(`variant_NN.tbx` — не менее 15 терминов, `variant_NN.tmx` — не менее 10
предложений) вы создаёте самостоятельно: либо аналогичным кодом, либо выгрузкой
из Smartcat, либо вручную.

In [41]:
DEMO_TBX = """<?xml version="1.0" encoding="UTF-8"?>
<martif type="TBX" xml:lang="en">
  <martifHeader><fileDesc><titleStmt><title>Demo glossary</title></titleStmt>
  </fileDesc></martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en">
        <tig><term>flight controller</term>
        <termNote type="partOfSpeech">noun</termNote>
        <descrip type="definition">Onboard flight computer.</descrip></tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp><term>полётный контроллер</term></termGrp></ntig>
      </langSet>
    </termEntry>
    <termEntry id="tid-002">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>payload</term>
        <termNote type="partOfSpeech">noun</termNote></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>целевая нагрузка</term>
        </termGrp></ntig></langSet>
    </termEntry>
    <termEntry id="tid-003">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>telemetry link</term></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>канал телеметрии</term>
        </termGrp></ntig></langSet>
    </termEntry>
  </body></text>
</martif>
"""

DEMO_TMX = """<?xml version="1.0" encoding="UTF-8"?>
<tmx version="1.4">
  <header creationtool="demo" creationtoolversion="1.0" segtype="sentence"
          o-tmf="TMX" adminlang="en" srclang="en" datatype="plaintext"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The flight controller stabilises the aircraft.</seg></tuv>
      <tuv xml:lang="ru"><seg>Полётный контроллер стабилизирует аппарат.</seg></tuv>
    </tu>
    <tu tuid="2">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The operator checks the telemetry link.</seg></tuv>
      <tuv xml:lang="ru"><seg>Оператор проверяет канал телеметрии.</seg></tuv>
    </tu>
    <tu tuid="3">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The payload is mounted under the fuselage.</seg></tuv>
      <tuv xml:lang="ru"><seg>Целевая нагрузка устанавливается под фюзеляжем.</seg></tuv>
    </tu>
  </body>
</tmx>
"""

with open("demo.tbx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TBX)
with open("demo.tmx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TMX)

print("Созданы файлы:", [f for f in os.listdir(".") if f.endswith((".tbx", ".tmx"))])

Созданы файлы: ['demo.tbx', 'demo.tmx', 'variant_10.tbx', 'variant_10.tmx']


## Блок 4. Задание — реализуйте четыре функции

Требования ко всем функциям: типизация из `typing`, соответствие PEP 8,
обработка отсутствующих элементов без падения (`None`-проверки).

**Подсказки:**

* язык секции читается как `element.get(XML_LANG)`, а не `element.get("lang")`;
* обход `<term>` внутри `langSet` удобно делать через `lang_set.iter("term")` —
  это покрывает и `tig`, и `ntig/termGrp`;
* текст сегмента лучше собирать через `"".join(seg.itertext())` — так не
  теряется текст при наличии инлайновых тегов;
* при подстановке терминов сортируйте их по убыванию длины.

In [42]:
def parse_tbx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TBX-глоссарий в плоскую таблицу терминологических пар."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    root = ET.parse(file_path).getroot()
    rows: List[Dict[str, str]] = []

    for entry in root.iter("termEntry"):
        subject = ""
        descrip = entry.find("descrip[@type='subjectField']")
        if descrip is not None and descrip.text:
            subject = descrip.text.strip()

        by_lang: Dict[str, Dict[str, str]] = {}
        for lang_set in entry.findall("langSet"):
            lang = (lang_set.get(XML_LANG) or "").split("-")[0].lower()
            if not lang:
                continue

            terms = [t.text.strip() for t in lang_set.iter("term") if (t.text or "").strip()]
            if not terms:
                continue

            note = lang_set.find(".//termNote[@type='partOfSpeech']")
            definition = lang_set.find(".//descrip[@type='definition']")

            by_lang[lang] = {
                "term": terms[0],
                "pos": (note.text or "").strip() if note is not None else "",
                "definition": (definition.text or "").strip() if definition is not None else "",
            }

        if src_lang in by_lang and tgt_lang in by_lang:
            rows.append({
                "entry_id": entry.get("id", ""),
                "source_term": by_lang[src_lang]["term"],
                "target_term": by_lang[tgt_lang]["term"],
                "pos": by_lang[src_lang]["pos"] or by_lang[tgt_lang]["pos"],
                "definition": by_lang[src_lang]["definition"],
                "subject_field": subject,
            })

    return pd.DataFrame(rows, columns=[
        "entry_id", "source_term", "target_term", "pos", "definition", "subject_field"
    ])


def parse_tmx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TMX-память переводов в таблицу параллельных сегментов."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    root = ET.parse(file_path).getroot()

    def _seg_text(tuv) -> str:
        seg = tuv.find("seg")
        if seg is None:
            return ""
        return re.sub(r"\s+", " ", "".join(seg.itertext())).strip()

    rows: List[Dict[str, Any]] = []

    for order, tu in enumerate(root.iter("tu"), start=1):
        segments: Dict[str, str] = {}
        for tuv in tu.findall("tuv"):
            lang = (tuv.get(XML_LANG) or tuv.get("lang") or "").split("-")[0].lower()
            text = _seg_text(tuv)
            if lang and text:
                segments[lang] = text

        if src_lang not in segments or tgt_lang not in segments:
            continue

        domain_node = tu.find("prop[@type='domain']")
        rows.append({
            "tuid": tu.get("tuid", str(order)),
            "source_segment": segments[src_lang],
            "target_segment": segments[tgt_lang],
            "domain": (domain_node.text or "").strip() if domain_node is not None else "",
            "src_len": len(segments[src_lang]),
        })

    frame = pd.DataFrame(rows, columns=[
        "tuid", "source_segment", "target_segment", "domain", "src_len"
    ])
    frame["prev_source"] = frame["source_segment"].shift(1).fillna("")
    return frame


def calculate_fuzzy_match(source_query: str, tm_df: pd.DataFrame,
                          threshold: float = 75.0,
                          context_before: Optional[str] = None,
                          top_n: int = 5) -> List[Dict[str, Any]]:
    """Найти в памяти переводов сегменты, похожие на запрос."""
    if tm_df is None or tm_df.empty or not source_query:
        return []

    query_norm = normalize(source_query)
    results: List[Dict[str, Any]] = []

    for _, row in tm_df.iterrows():
        seg_norm = normalize(row["source_segment"])
        lev = levenshtein_ratio(query_norm, seg_norm)
        tsr = token_sort_ratio(source_query, row["source_segment"])
        score = max(lev, tsr)

        if score < threshold:
            continue

        if score >= 100.0:
            ctx_ok = (context_before is not None
                      and normalize(context_before) == normalize(str(row.get("prev_source", ""))))
            kind = "ICE (101%)" if ctx_ok else "Exact (100%)"
        elif score >= 95.0:
            kind = "High Fuzzy"
        elif score >= 75.0:
            kind = "Fuzzy"
        else:
            kind = "Low Fuzzy"

        results.append({
            "tuid": row["tuid"],
            "source_segment": row["source_segment"],
            "target_segment": row["target_segment"],
            "score": score,
            "lev_ratio": lev,
            "token_sort_ratio": tsr,
            "match_type": kind,
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_n]


def apply_term_glossary(text: str, glossary_df: pd.DataFrame,
                        template: str = "{source} [{target}]") -> str:
    """Разметить исходный текст переводными эквивалентами из глоссария."""
    if glossary_df is None or glossary_df.empty or not text:
        return text

    pairs = sorted(
        zip(glossary_df["source_term"], glossary_df["target_term"]),
        key=lambda p: len(p[0]),
        reverse=True,
    )

    protected: List[str] = []

    def stash(repl: str) -> str:
        protected.append(repl)
        return f"\x00{len(protected) - 1}\x00"

    result = text
    for src, tgt in pairs:
        pattern = re.compile(rf"(?<!\w){re.escape(src)}(?!\w)", re.IGNORECASE)
        result = pattern.sub(
            lambda m: stash(template.format(source=m.group(0), target=tgt)), result
        )

    return re.sub(r"\x00(\d+)\x00", lambda m: protected[int(m.group(1))], result)

### 4.1. Вспомогательные функции (реализуйте при необходимости)

Эти функции не оцениваются отдельно, но нужны для `calculate_fuzzy_match`.

In [36]:
def normalize(text: str) -> str:
    """Привести строку к единому виду для корректного сравнения."""
    text = unicodedata.normalize("NFKC", text).lower().replace("ё", "е")
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


def levenshtein_ratio(a: str, b: str) -> float:
    """Вычислить схожесть двух строк (в процентах) через расстояние редактирования."""
    if not a and not b:
        return 100.0
    longest = max(len(a), len(b))
    return round((1.0 - edit_distance(a, b) / longest) * 100.0, 2)


def token_sort_ratio(a: str, b: str) -> float:
    """Оценить схожесть строк, не учитывая порядок слов в предложении."""
    sorted_a = " ".join(sorted(normalize(a).split()))
    sorted_b = " ".join(sorted(normalize(b).split()))
    return levenshtein_ratio(sorted_a, sorted_b)

## Блок 5. Автотесты

Запустите ячейки ниже **после** реализации функций. Все проверки должны
завершиться без `AssertionError`. Тесты работают на демо-файлах из Блока 3.

In [43]:
# --- Тест 1: метрики сходства -------------------------------------------
assert abs(levenshtein_ratio("telemetry", "telemetry") - 100.0) < 1e-6, \
    "Одинаковые строки должны давать 100 %"
assert levenshtein_ratio("", "") == 100.0, "Две пустые строки — 100 %"
assert 0.0 <= levenshtein_ratio("payload", "airframe") < 50.0, \
    "Непохожие строки должны давать низкий процент"
assert abs(token_sort_ratio("angle of attack", "attack of angle") - 100.0) < 1e-6, \
    "Token Sort Ratio должен игнорировать порядок слов"
assert normalize("  Angle   OF ATTACK, critical! ") == "angle of attack critical", \
    "normalize: нижний регистр, без пунктуации, одиночные пробелы"
print("Тест 1 (метрики сходства) — OK")

Тест 1 (метрики сходства) — OK


In [44]:
# --- Тест 2: парсинг TBX -------------------------------------------------
glossary_df = parse_tbx("demo.tbx")

assert isinstance(glossary_df, pd.DataFrame), "parse_tbx должен вернуть DataFrame"
assert len(glossary_df) == 3, f"Ожидается 3 термина, получено {len(glossary_df)}"
for column in ["entry_id", "source_term", "target_term", "pos",
               "definition", "subject_field"]:
    assert column in glossary_df.columns, f"Нет колонки '{column}'"

row = glossary_df.loc[glossary_df["source_term"] == "flight controller"].iloc[0]
assert row["target_term"] == "полётный контроллер", "Неверный русский эквивалент"
assert row["entry_id"] == "tid-001", "Не извлечён атрибут id у termEntry"
assert row["pos"] == "noun", "Не извлечён termNote[@type='partOfSpeech']"
assert "telemetry link" in set(glossary_df["source_term"]), \
    "Термин из <tig> без termNote должен быть извлечён"
print("Тест 2 (парсинг TBX) — OK")
glossary_df.head()

Тест 2 (парсинг TBX) — OK


,entry_id,source_term,target_term,pos,definition,subject_field
0,tid-001,flight controller,полётный контроллер,noun,Onboard flight computer.,Demo
1,tid-002,payload,целевая нагрузка,noun,,Demo
2,tid-003,telemetry link,канал телеметрии,,,Demo


In [45]:
# --- Тест 3: парсинг TMX -------------------------------------------------
tm_df = parse_tmx("demo.tmx")

assert isinstance(tm_df, pd.DataFrame), "parse_tmx должен вернуть DataFrame"
assert len(tm_df) == 3, f"Ожидается 3 единицы перевода, получено {len(tm_df)}"
for column in ["tuid", "source_segment", "target_segment", "domain",
               "src_len", "prev_source"]:
    assert column in tm_df.columns, f"Нет колонки '{column}'"

first = tm_df.iloc[0]
assert first["source_segment"].startswith("The flight controller"), \
    "Не извлечён текст <seg> исходного языка"
assert first["target_segment"].startswith("Полётный контроллер"), \
    "Не извлечён текст <seg> языка перевода"
assert first["prev_source"] == "", "У первой строки prev_source должен быть пустым"
assert tm_df.iloc[1]["prev_source"] == first["source_segment"], \
    "prev_source должен содержать предыдущий исходный сегмент"
assert first["src_len"] == len(first["source_segment"]), "Неверная длина сегмента"
print("Тест 3 (парсинг TMX) — OK")
tm_df.head()

Тест 3 (парсинг TMX) — OK


,tuid,source_segment,target_segment,domain,src_len,prev_source
0,1,The flight controller stabilises the aircraft.,Полётный контроллер стабилизирует аппарат.,Demo,46,
1,2,The operator checks the telemetry link.,Оператор проверяет канал телеметрии.,Demo,39,The flight controller stabilises the aircraft.
2,3,The payload is mounted under the fuselage.,Целевая нагрузка устанавливается под фюзеляжем.,Demo,42,The operator checks the telemetry link.


In [46]:
# --- Тест 4: Fuzzy Match и ICE ------------------------------------------
exact_query = "The operator checks the telemetry link."
matches = calculate_fuzzy_match(exact_query, tm_df, threshold=70.0)
assert matches, "Точное совпадение обязано быть найдено"
assert abs(matches[0]["score"] - 100.0) < 1e-6, "Точное совпадение — 100 %"
assert matches[0]["match_type"] == "Exact (100%)", \
    "Без контекста 100 % — это Exact, а не ICE"

ice = calculate_fuzzy_match(
    exact_query, tm_df, threshold=70.0,
    context_before="The flight controller stabilises the aircraft.")
assert ice[0]["match_type"] == "ICE (101%)", \
    "При совпадении предыдущего сегмента 100 % должно стать ICE"

fuzzy = calculate_fuzzy_match("The engineer checks the telemetry link.",
                              tm_df, threshold=70.0)
assert fuzzy, "Похожий сегмент должен попадать в выдачу"
assert 75.0 <= fuzzy[0]["score"] < 100.0, \
    f"Ожидается Fuzzy 75–99 %, получено {fuzzy[0]['score']}"
assert fuzzy[0]["match_type"] in {"Fuzzy", "High Fuzzy"}, "Неверный тип совпадения"

scores = [m["score"] for m in calculate_fuzzy_match("telemetry link check",
                                                    tm_df, threshold=0.0)]
assert scores == sorted(scores, reverse=True), \
    "Результаты должны быть отсортированы по убыванию сходства"
assert calculate_fuzzy_match("Совершенно посторонний текст про бухгалтерию.",
                             tm_df, threshold=75.0) == [], \
    "Ниже порога результатов быть не должно"
assert len(calculate_fuzzy_match("telemetry", tm_df, threshold=0.0, top_n=2)) <= 2, \
    "Параметр top_n не учитывается"
print("Тест 4 (Fuzzy Match и ICE) — OK")

Тест 4 (Fuzzy Match и ICE) — OK


In [47]:
# --- Тест 5: подстановка терминов ---------------------------------------
marked = apply_term_glossary("The flight controller reads the payload status.",
                             glossary_df)
assert "[полётный контроллер]" in marked, "Термин из глоссария не подставлен"
assert "[целевая нагрузка]" in marked, "Подставлен не каждый найденный термин"
assert "flight controller" in marked, "Исходный термин должен сохраняться"

assert apply_term_glossary("No terms here at all.", glossary_df) == \
    "No terms here at all.", "Текст без терминов не должен меняться"

case_test = apply_term_glossary("Payload mass is 2 kg.", glossary_df)
assert "Payload [целевая нагрузка]" in case_test, \
    "Поиск должен быть регистронезависимым, а регистр текста — сохраняться"

assert "[целевая нагрузка]" not in apply_term_glossary("payloads are heavy",
                                                       glossary_df), \
    "Совпадение должно быть по границам слов, а не по подстроке"
print("Тест 5 (подстановка терминов) — OK")
print(marked)

Тест 5 (подстановка терминов) — OK
The flight controller [полётный контроллер] reads the payload [целевая нагрузка] status.


## Блок 6. Работа со своим вариантом

После прохождения всех тестов выполните задание на данных своего варианта.

In [48]:
def indent_tree(element, level: int = 0, space: str = "  ") -> None:
    """Рекурсивно расставить отступы в XML-дереве (работает и с lxml, и с ET)."""
    pad = "\n" + space * level
    if len(element):
        if not (element.text or "").strip():
            element.text = pad + space
        for child in element:
            indent_tree(child, level + 1, space)
        if not (element.tail or "").strip():
            element.tail = pad
        if not (element[-1].tail or "").strip():
            element[-1].tail = pad
    elif level and not (element.tail or "").strip():
        element.tail = pad


TERMS: List[Dict[str, str]] = [
    # --- 100 терминов корпуса (en, ru, pos, definition_en) ---
    {"en": "charging", "ru": "зарядка", "pos": "noun", "def_en": "The process of supplying electrical energy to a battery or an electric vehicle."},
    {"en": "observatory", "ru": "обсерватория", "pos": "noun", "def_en": "An institution that monitors and analyzes data in a specific field, such as alternative fuels."},
    {"en": "infrastructure", "ru": "инфраструктура", "pos": "noun", "def_en": "The basic physical and organizational structures and facilities needed for the operation of a system."},
    {"en": "charging station", "ru": "зарядная станция", "pos": "noun", "def_en": "A stationary device designed to recharge the traction batteries of electric vehicles."},
    {"en": "key performance indicator", "ru": "показатель эффективности", "pos": "noun", "def_en": "A quantitative metric used to evaluate the performance or success of a process or system."},
    {"en": "target value", "ru": "целевое значение", "pos": "noun", "def_en": "A specified quantitative value that must be achieved as a result of a policy or project."},
    {"en": "charging network", "ru": "зарядная сеть", "pos": "noun", "def_en": "An interconnected system of charging stations and points managed by operators."},
    {"en": "standard", "ru": "стандарт", "pos": "noun", "def_en": "A normative document that establishes requirements for products, processes, or services."},
    {"en": "term", "ru": "термин", "pos": "noun", "def_en": "A word or phrase used to designate a strictly defined concept in a particular field of knowledge."},
    {"en": "alternating current", "ru": "переменный ток", "pos": "noun", "def_en": "An electric current whose direction and magnitude periodically reverse over time."},
    {"en": "direct current", "ru": "постоянный ток", "pos": "noun", "def_en": "An electric current that flows in one direction only and does not change its direction over time."},
    {"en": "power", "ru": "мощность", "pos": "noun", "def_en": "A physical quantity characterizing the rate of transfer or conversion of electrical energy."},
    {"en": "charging capacity", "ru": "зарядная способность", "pos": "noun", "def_en": "The maximum level of power or energy that a charger can deliver to an electric vehicle."},
    {"en": "connector", "ru": "разъем", "pos": "noun", "def_en": "An electrical connector providing mechanical and electrical connection between a charging station and a vehicle."},
    {"en": "interoperability", "ru": "интероперабельность", "pos": "noun", "def_en": "The ability of different systems and devices to exchange data and operate together without user intervention."},
    {"en": "carmaker", "ru": "автопроизводитель", "pos": "noun", "def_en": "A company engaged in the design and manufacture of automobiles."},
    {"en": "consolidation", "ru": "консолидация", "pos": "noun", "def_en": "The process of combining separate technological solutions or market participants into a single stable system."},
    {"en": "megawatt charging", "ru": "мегаваттная зарядка", "pos": "noun", "def_en": "An ultra-fast charging technology with a capacity above 1 MW for heavy-duty transport."},
    {"en": "charging power", "ru": "зарядная мощность", "pos": "noun", "def_en": "The amount of electrical power delivered by a charger to a battery."},
    {"en": "software", "ru": "программное обеспечение", "pos": "noun", "def_en": "A set of programs and data that ensures the operation of charging hardware."},
    {"en": "charging standard", "ru": "стандарт зарядки", "pos": "noun", "def_en": "An agreed set of technical requirements for the charging process and equipment."},
    {"en": "standardization", "ru": "стандартизация", "pos": "noun", "def_en": "The activity of establishing uniform rules and norms to ensure compatibility and quality."},
    {"en": "charging function", "ru": "функция зарядки", "pos": "noun", "def_en": "The purpose or role of charging equipment in a specific operational environment."},
    {"en": "public charging", "ru": "общественная зарядка", "pos": "noun", "def_en": "Charging available to all electric vehicle drivers without restrictions on the user group."},
    {"en": "private charging", "ru": "частная зарядка", "pos": "noun", "def_en": "Charging carried out on private territory for personal use or for fleet needs."},
    {"en": "deployment", "ru": "развертывание", "pos": "noun", "def_en": "The process of commissioning and scaling up charging infrastructure."},
    {"en": "unrestricted access", "ru": "неограниченный доступ", "pos": "noun", "def_en": "The absence of temporal, spatial, or user restrictions on the use of a charging station."},
    {"en": "semi-public charging", "ru": "полуобщественная зарядка", "pos": "noun", "def_en": "Charging available to a broad audience but located on private territory and subject to restrictions."},
    {"en": "access restriction", "ru": "ограничение доступа", "pos": "noun", "def_en": "A condition under which the use of a charging station is possible only for certain categories of users."},
    {"en": "charging service", "ru": "зарядный сервис", "pos": "noun", "def_en": "A service providing access to charging infrastructure and related functions."},
    {"en": "navigation service", "ru": "навигационный сервис", "pos": "noun", "def_en": "A software and hardware complex that provides search and routing to charging stations."},
    {"en": "publicly accessible infrastructure", "ru": "общедоступная инфраструктура", "pos": "noun", "def_en": "Charging stations that can be used by all electric vehicle drivers."},
    {"en": "private infrastructure", "ru": "частная инфраструктура", "pos": "noun", "def_en": "A set of charging devices that are privately owned and not available to the general public."},
    {"en": "strategy", "ru": "стратегия", "pos": "noun", "def_en": "A long-term action plan aimed at achieving goals in the development of charging infrastructure."},
    {"en": "infrastructure development", "ru": "развитие инфраструктуры", "pos": "noun", "def_en": "The process of quantitative and qualitative growth of the charging network."},
    {"en": "fast charging", "ru": "быстрая зарядка", "pos": "noun", "def_en": "High-power direct current charging that reduces charging time."},
    {"en": "slow charging", "ru": "медленная зарядка", "pos": "noun", "def_en": "Low-power alternating current charging, typically used during long-term parking."},
    {"en": "business model", "ru": "бизнес-модель", "pos": "noun", "def_en": "A scheme for organizing the commercial activity of a charging infrastructure operator."},
    {"en": "charging cost", "ru": "стоимость зарядки", "pos": "noun", "def_en": "The total cost to the user of purchasing electricity for charging an electric vehicle."},
    {"en": "convenience", "ru": "удобство", "pos": "noun", "def_en": "The degree of comfort and ease of use of charging infrastructure for the driver."},
    {"en": "smart charging", "ru": "интеллектуальная зарядка", "pos": "noun", "def_en": "A technology for managing the charging process taking into account grid load and tariffs."},
    {"en": "home charging", "ru": "домашняя зарядка", "pos": "noun", "def_en": "Charging an electric vehicle at the owner's place of residence."},
    {"en": "demand", "ru": "спрос", "pos": "noun", "def_en": "The need of users for charging services, expressed in quantitative terms."},
    {"en": "user satisfaction", "ru": "удовлетворенность пользователей", "pos": "noun", "def_en": "A subjective assessment of the quality of charging services by electric vehicle drivers."},
    {"en": "charging market", "ru": "рынок зарядки", "pos": "noun", "def_en": "The sphere of economic relations related to the provision of electric vehicle charging services."},
    {"en": "integration", "ru": "интеграция", "pos": "noun", "def_en": "The combination of charging infrastructure with other elements of transport and energy systems."},
    {"en": "investment", "ru": "капиталовложение", "pos": "noun", "def_en": "A long-term investment of financial resources in the creation and development of charging infrastructure."},
    {"en": "public contract", "ru": "публичный контракт", "pos": "noun", "def_en": "An agreement between a public authority and a private party on the creation of charging infrastructure."},
    {"en": "joint venture", "ru": "совместное предприятие", "pos": "noun", "def_en": "A commercial organization created by two or more parties for joint activity."},
    {"en": "concession", "ru": "концессия", "pos": "noun", "def_en": "An agreement on transferring to a private partner the rights to create and operate charging infrastructure."},
    {"en": "risk", "ru": "риск", "pos": "noun", "def_en": "The probability of losses or failure to fulfill obligations under a project."},
    {"en": "financing", "ru": "финансирование", "pos": "noun", "def_en": "The provision of funds for the implementation of a charging infrastructure project."},
    {"en": "availability-based contract", "ru": "эксплуатационная готовность", "pos": "noun", "def_en": "The state of infrastructure in which it is capable of performing its specified functions."},
    {"en": "payment", "ru": "платеж", "pos": "noun", "def_en": "Funds transferred by one party to a contract to the other party."},
    {"en": "license", "ru": "лицензия", "pos": "noun", "def_en": "A permit to carry out activities for the creation and operation of charging infrastructure."},
    {"en": "concession model", "ru": "концессионная модель", "pos": "noun", "def_en": "A scheme for organizing public-private partnership in the creation of infrastructure."},
    {"en": "company", "ru": "компания", "pos": "noun", "def_en": "A legal entity engaged in commercial activity in the charging market."},
    {"en": "energy company", "ru": "энергетическая компания", "pos": "noun", "def_en": "An organization engaged in the production, distribution, or supply of electricity."},
    {"en": "market model", "ru": "рыночная модель", "pos": "noun", "def_en": "A set of principles for organizing interaction among participants in the charging market."},
    {"en": "service provider", "ru": "поставщик услуг", "pos": "noun", "def_en": "An organization providing services for access to charging infrastructure."},
    {"en": "certification", "ru": "сертификация", "pos": "noun", "def_en": "A procedure for confirming the conformity of products with established requirements."},
    {"en": "coordination", "ru": "координация", "pos": "noun", "def_en": "The alignment of actions of market participants to achieve common goals."},
    {"en": "protocol", "ru": "протокол", "pos": "noun", "def_en": "A set of rules and data exchange formats between charging infrastructure devices."},
    {"en": "regulation", "ru": "регламент", "pos": "noun", "def_en": "A normative legal act establishing binding requirements."},
    {"en": "harmonization", "ru": "гармонизация", "pos": "noun", "def_en": "Bringing technical norms and rules to a common form to ensure compatibility."},
    {"en": "regulatory framework", "ru": "нормативно-правовая база", "pos": "noun", "def_en": "A set of laws and regulations governing activities in the field of charging."},
    {"en": "innovation", "ru": "инновация", "pos": "noun", "def_en": "The introduction of new technologies or solutions in the field of charging infrastructure."},
    {"en": "openness", "ru": "открытость", "pos": "noun", "def_en": "The principle of non-discriminatory access of participants to the market and technologies."},
    {"en": "concept", "ru": "понятие", "pos": "noun", "def_en": "A form of thinking reflecting the general essential properties of objects."},
    {"en": "cost", "ru": "затрата", "pos": "noun", "def_en": "The consumption of resources (financial, material, labor) for the creation and operation of infrastructure."},
    {"en": "scaling", "ru": "масштабирование", "pos": "noun", "def_en": "The process of increasing the coverage and capacity of charging infrastructure."},
    {"en": "competitive environment", "ru": "конкурентная среда", "pos": "noun", "def_en": "A set of conditions in which participants in the charging market operate."},
    {"en": "tool", "ru": "инструмент", "pos": "noun", "def_en": "A tool or software product used to ensure system compatibility."},
    {"en": "barrier", "ru": "барьер", "pos": "noun", "def_en": "An obstacle to the spread of technologies or market entry."},
    {"en": "sector", "ru": "сектор", "pos": "noun", "def_en": "An area of the economy related to the production and servicing of electric vehicles and charging infrastructure."},
    {"en": "policymaker", "ru": "директивный орган", "pos": "noun", "def_en": "A state structure that develops and implements policy in the field of transport and energy."},
    {"en": "network architecture", "ru": "архитектура сети", "pos": "noun", "def_en": "The structural organization of components of an energy or charging network."},
    {"en": "hardware layer", "ru": "аппаратный уровень", "pos": "noun", "def_en": "The level of interaction of physical components of charging infrastructure."},
    {"en": "communication layer", "ru": "коммуникационный уровень", "pos": "noun", "def_en": "The level of data exchange between devices and management systems."},
    {"en": "information layer", "ru": "информационный уровень", "pos": "noun", "def_en": "The level of representation and interpretation of data in a charging system."},
    {"en": "service layer", "ru": "сервисный уровень", "pos": "noun", "def_en": "The level of provision of user services based on standardized interfaces."},
    {"en": "business layer", "ru": "бизнес-уровень", "pos": "noun", "def_en": "The level of organizing commercial relations between market participants."},
    {"en": "degree of interoperability", "ru": "степень интероперабельности", "pos": "noun", "def_en": "The level of system compatibility sufficient for joint operation."},
    {"en": "charging point operator", "ru": "оператор зарядных станций", "pos": "noun", "def_en": "An organization managing charging stations and providing access to them."},
    {"en": "payment system", "ru": "платежная система", "pos": "noun", "def_en": "A set of technical and software tools for making payments for charging services."},
    {"en": "advantage", "ru": "преимущество", "pos": "noun", "def_en": "A positive property or benefit from using a particular model."},
    {"en": "concept of roaming", "ru": "концепция роуминга", "pos": "noun", "def_en": "A system of views or a theoretical construct describing the roaming phenomenon."},
    {"en": "roaming", "ru": "роуминг", "pos": "noun", "def_en": "A service allowing a user to charge in networks of different operators under a single contract."},
    {"en": "access method", "ru": "метод доступа", "pos": "noun", "def_en": "A method of authorization and payment when connecting to a charging station."},
    {"en": "grid congestion", "ru": "перегрузка сети", "pos": "noun", "def_en": "A state of the power grid in which the load exceeds permissible values."},
    {"en": "structured market model", "ru": "структурированная рыночная модель", "pos": "noun", "def_en": "A structured description of interaction among market participants."},
    {"en": "overview", "ru": "обзор", "pos": "noun", "def_en": "A brief systematized presentation of information on a specific topic."},
    {"en": "connection", "ru": "подключение", "pos": "noun", "def_en": "The process of establishing a connection between devices or systems."},
    {"en": "pattern", "ru": "схема", "pos": "noun", "def_en": "A sequence of stages or a structure for organizing a process."},
    {"en": "customer journey", "ru": "поездка клиента", "pos": "noun", "def_en": "A model of the sequence of user actions when interacting with charging infrastructure."},
    {"en": "EV driver", "ru": "водитель электромобиля", "pos": "noun", "def_en": "A user of a vehicle who charges and operates it."},
    {"en": "selection criterion", "ru": "критерий выбора", "pos": "noun", "def_en": "A characteristic on the basis of which a user decides on the choice of a charging station."},
    {"en": "quality of service", "ru": "качество обслуживания", "pos": "noun", "def_en": "A set of characteristics of charging services that determine their ability to meet user needs."},
    {"en": "reliability", "ru": "надежность", "pos": "noun", "def_en": "The ability of a charging station to perform its functions over a specified time without failures."},
    {"en": "up-time", "ru": "время безотказной работы", "pos": "noun", "def_en": "An indicator reflecting the proportion of time during which equipment is in an operable state."},
]

SUBJECT_FIELD = "Автомобилестроение и электротранспорт"
TBX_PATH = "variant_10.tbx"


def build_tbx(terms: Sequence[Dict[str, str]], path: str,
              subject_field: str = SUBJECT_FIELD) -> str:
    """Сгенерировать валидный TBX-файл (TBX-Core v2, martif)."""
    martif = ET.Element("martif", {"type": "TBX", XML_LANG: "en"})

    header = ET.SubElement(martif, "martifHeader")
    file_desc = ET.SubElement(header, "fileDesc")
    title_stmt = ET.SubElement(file_desc, "titleStmt")
    ET.SubElement(title_stmt, "title").text = f"Glossary EN-RU: {subject_field}"
    source_desc = ET.SubElement(file_desc, "sourceDesc")
    ET.SubElement(source_desc, "p").text = "Lab 01, variant 10, generated automatically"
    encoding_desc = ET.SubElement(header, "encodingDesc")
    ET.SubElement(encoding_desc, "p", {"type": "XCSURI"}).text = "TBXXCSV02.xcs"

    body = ET.SubElement(ET.SubElement(martif, "text"), "body")

    for index, item in enumerate(terms, start=1):
        entry = ET.SubElement(body, "termEntry", {"id": f"tid-{index:03d}"})
        ET.SubElement(entry, "descrip", {"type": "subjectField"}).text = subject_field

        lang_en = ET.SubElement(entry, "langSet", {XML_LANG: "en"})
        tig = ET.SubElement(lang_en, "tig")
        ET.SubElement(tig, "term").text = item["en"]
        ET.SubElement(tig, "termNote", {"type": "partOfSpeech"}).text = item["pos"]
        ET.SubElement(tig, "descrip", {"type": "definition"}).text = item["def_en"]

        lang_ru = ET.SubElement(entry, "langSet", {XML_LANG: "ru"})
        term_grp = ET.SubElement(ET.SubElement(lang_ru, "ntig"), "termGrp")
        ET.SubElement(term_grp, "term").text = item["ru"]
        ET.SubElement(term_grp, "termNote", {"type": "partOfSpeech"}).text = item["pos"]

    indent_tree(martif)
    xml_body = ET.tostring(martif, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write('<!DOCTYPE martif SYSTEM "TBXcoreStructV02.dtd">\n')
        handler.write(xml_body)
    return path


build_tbx(TERMS, TBX_PATH)
print(f"Создан {TBX_PATH}: {os.path.getsize(TBX_PATH)} байт, терминов: {len(TERMS)}\n")
print("--- первые 26 строк файла ---")
with open(TBX_PATH, encoding="utf-8") as handler:
    for line in list(handler)[:26]:
        print(line.rstrip())

Создан variant_10.tbx: 73813 байт, терминов: 100

--- первые 26 строк файла ---
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE martif SYSTEM "TBXcoreStructV02.dtd">
<martif type="TBX" xml:lang="en">
  <martifHeader>
    <fileDesc>
      <titleStmt>
        <title>Glossary EN-RU: Автомобилестроение и электротранспорт</title>
      </titleStmt>
      <sourceDesc>
        <p>Lab 01, variant 10, generated automatically</p>
      </sourceDesc>
    </fileDesc>
    <encodingDesc>
      <p type="XCSURI">TBXXCSV02.xcs</p>
    </encodingDesc>
  </martifHeader>
  <text>
    <body>
      <termEntry id="tid-001">
        <descrip type="subjectField">Автомобилестроение и электротранспорт</descrip>
        <langSet xml:lang="en">
          <tig>
            <term>charging</term>
            <termNote type="partOfSpeech">noun</termNote>
            <descrip type="definition">The process of supplying electrical energy to a battery or an electric vehicle.</descrip>
          </tig>


In [49]:
PARALLEL_CORPUS: List[Tuple[str, str]] = [
    ("In 2022, more than 600,000 public slow charging points were installed globally.",
     "В 2022 году во всем мире было установлено более 600 000 общественных пунктов медленной зарядки."),
    ("The European Alternative Fuels Observatory provides a reliable benchmark for understanding these elements.",
     "Европейская обсерватория альтернативных видов топлива предусматривает надежные целевые ориентиры для понимания этих аспектов."),
    ("Adequate charging infrastructure is an essential component of the ecosystem.",
     "Одним из важнейших элементов экосистемы служит адекватная зарядная инфраструктура."),
    ("Even within a country, the required amount of charging stations will differ per region.",
     "Даже в пределах одной страны требующееся число зарядных станций будет различаться по регионам."),
    ("To establish a definition for 'sufficient' charging infrastructure, Key Performance Indicators have been developed.",
     "Для определения «достаточной» зарядной инфраструктуры были разработаны ключевые показатели эффективности."),
    ("When establishing specific target values for the aforementioned KPI's, it is essential to consider them.",
     "Конкретные целевые значения для упомянутых выше КПЭ крайне важно учитывать."),
    ("A charging network with limited interoperability can pose challenges.",
     "Зарядная сеть с ограниченной интероперабельностью может создавать проблемы."),
    ("This section provides an overview of the terms and definitions related to charging stations, and the standards adopted.",
     "В настоящем разделе представлен обзор терминов и определений, относящихся к зарядным станциям, а также принятых стандартов."),
    ("In this study, we use the more general term \"charging\".",
     "В настоящем исследовании использован более общий термин «зарядка»."),
    ("An AC charging station directly provides AC to the EV.",
     "Станция зарядки переменным током подает переменный ток напрямую в ЭМ."),
    ("Any DC charging station converts AC from the grid to DC.",
     "Любая станция зарядки постоянным током преобразует переменный ток из сети в постоянный."),
    ("DC chargers can provide higher capacities directly to the EV.",
     "Устройства зарядки постоянным током могут обеспечивать более высокие значения мощности непосредственно для ЭМ."),
    ("It makes sense to distinguish between charging capacities.",
     "Помимо определения пунктов зарядки, имеет смысл провести различие между зарядными способностями."),
    ("In the European Union, the Type 2 connector has been prescribed for AC charging.",
     "В Европейском союзе для зарядки переменным током предписан разъем типа 2."),
    ("High power recharging points shall be equipped, for interoperability purposes, with connectors of Type 2.",
     "Высокомощные пункты подзарядки электромобилей оснащаются для обеспечения интероперабельности разъемами типа 2."),
    ("Some carmakers have recently chosen to adopt the Tesla DC standard for their vehicles.",
     "Некоторые автопроизводители недавно решили принять стандарт постоянного тока «Теслы» для своих автомобилей."),
    ("Apparently, even on hardware interoperability, consolidation is still taking place.",
     "Очевидно, что консолидация происходит даже по аспектам интероперабельности аппаратного обеспечения."),
    ("A new Megawatt Charging Standard is in development for heavy-duty vehicles.",
     "В настоящее время разрабатывается новый стандарт мегаваттной зарядки для транспортных средств большой грузоподъемности."),
    ("This initiative provides heavy-duty trucks with a charging capacity potentially of almost 4MW.",
     "Эта инициатива обеспечивает тяжелые грузовики зарядной мощностью почти в 4 МВт."),
    ("The MCS concept ranges beyond a charging system and connector, but includes recommendations on software standards.",
     "Концепция мегаваттной зарядки не согласуется с зарядной системой и разъемами, но соответствует рекомендациям по стандартам программного обеспечения."),
    ("The European Union and the United States have cooperated to develop a shared vision on a standard for charging electric heavy-duty vehicles.",
     "Европейский союз и Соединенные Штаты Америки занимаются разработкой общего видения стандарта зарядки электромобилей большой грузоподъемности."),
    ("The alignment of our approaches to standardization will be critical for the roll-out of dedicated recharging infrastructure.",
     "Согласование подходов к стандартизации будет иметь решающее значение для развертывания специализированной зарядной инфраструктуры."),
    ("When defining the charging function, a distinction is often made between public and private charging.",
     "При определении функции зарядки зачастую проводится различие между общественными и частными вариантами зарядки."),
    ("Public charging refers to providing non-discriminatory access to charging.",
     "Общественный вариант подразумевает предоставление недискриминационного доступа к зарядке."),
    ("Private charging involves charging for personal use.",
     "Частный вариант зарядки предусматривает зарядку, предназначенную для личного пользования."),
    ("The deployment of public charging is largely affected by policy choices of local governments.",
     "Развертывание общественной зарядной системы в значительной степени зависит от выбора стратегий местными органами власти."),
    ("Access is considered to be unrestricted: every EV driver is able to charge at any period of time.",
     "Доступ считается неограниченным, когда каждый водитель ЭМ может заряжать его в любое время."),
    ("Semi-public charging is charging for public use on private territory.",
     "Полуобщественный вариант зарядки — это зарядка для общественного пользования на частной территории."),
    ("Restrictions may be present, either through access restrictions or restrictions in user groups.",
     "Ограничения могут быть предусмотрены на уровне доступа или групп пользователей."),
    ("This category is called 'semi-public' because the purpose is to provide a charging service.",
     "Данная категория обозначена как полуобщественная, так как она нацелена на предоставление зарядного сервиса."),
    ("These charging stations will not be displayed on any map or navigation service.",
     "Эти зарядные станции не будут указываться ни на одной из карт или в контексте навигационного сервиса."),
    ("The usage statistics often rely on publicly accessible charging infrastructure.",
     "Статистические данные об использовании зачастую основываются на учете общедоступной зарядной инфраструктуры."),
    ("Private charging can account for up to 75 per cent of the total charging network.",
     "На частную зарядную инфраструктуру может приходиться до 75 % всей зарядной сети."),
    ("Public government may want to develop policies that distinguish between public and semi-public charging station deployment.",
     "Государственные органы, возможно, пожелают разработать стратегии, предусматривающие различие между общественными и полуобщественными зарядными станциями."),
    ("A typical long-term strategy for charging infrastructure involves several steps.",
     "Обычная долгосрочная стратегия развития инфраструктуры зарядки предусматривает несколько этапов."),
    ("Deployment of a DC fast charging network to provide basic coverage.",
     "Развертывание сети быстрой зарядки постоянным током для обеспечения базового покрытия."),
    ("Deployment of an extensive AC charging network, allowing ubiquitous charging during long-stay parking.",
     "Развертывание широкой сети зарядки переменным током, обеспечивающей повсеместную зарядку во время продолжительной стоянки."),
    ("Extend use cases and business models to provide better services.",
     "Расширение вариантов использования и бизнес-моделей для предоставления более качественных услуг."),
    ("Cost: Home or depot charging is the preferred option for EV charging.",
     "Стоимость: зарядка на дому или на базе служит предпочтительным вариантом зарядки ЭМ."),
    ("Convenience: Home or depot charging enables overnight charging.",
     "Удобство: зарядка на дому или на базе позволяет заряжать аккумуляторную батарею в ночное время."),
    ("Grid impact: Slow charging is compatible with smart charging and vehicle-to-grid operations.",
     "Воздействие на сеть: медленная зарядка совместима с интеллектуальной зарядкой и подключением транспортного средства к общей электросети."),
    ("As the EV stock grows, a smaller proportion of owners will have access to home charging.",
     "С ростом парка ЭМ все меньшая доля их владельцев будет иметь доступ к зарядке на дому."),
    ("Establish a reliable functioning network of EV charging points with sufficient capacity to meet user demand.",
     "Создание надежно функционирующей сети пунктов зарядки ЭМ с мощностью, достаточной для удовлетворения спроса пользователей."),
    ("Achieve high levels of user satisfaction with public EV charging services.",
     "Достижение высокого уровня удовлетворенности пользователей общественным сервисом зарядки ЭМ."),
    ("Create a competitive market for EV charging that drives fair prices for users.",
     "Создание конкурентного рынка зарядки ЭМ, обеспечивающего справедливые цены для пользователей."),
    ("Integrate EV charging within a wider eco-mobility strategy.",
     "Интегрирование зарядки электромобилей в более широкую стратегию экономичности."),
    ("This requires a significant investment which need to be covered by private and/or public investments.",
     "Для этого требуются значительные капиталовложения, которые должны покрываться за счет частных и/или государственных средств."),
    ("Public contract: The public authority controls the installation and use of the infrastructure.",
     "Публичный контракт: государственный орган контролирует аспекты установки и использования инфраструктуры."),
    ("Joint venture: The public authority and private partner share control of the infrastructure.",
     "Совместное предприятие: государственный орган и частный партнер совместно контролируют инфраструктуру."),
    ("Concession: The public authority retains some control over the operation of the infrastructure.",
     "Концессия: государственный орган сохраняет определенный контроль за аспектами функционирования инфраструктуры."),
    ("The risks are typically transferred to the private partner.",
     "Все риски обычно передаются частному партнеру."),
    ("The private partner finances the capital and maintenance expenditure.",
     "Частный партнер финансирует капитальные затраты и расходы на техническое обслуживание."),
    ("Availability-based contract: the infrastructure must be continually available for its intended use.",
     "Контракт, основанный на эксплуатационной готовности: инфраструктура должна быть постоянно доступна для использования."),
    ("The private partner is paid by the public authority over the duration of the contract.",
     "Частный партнер получает платежи в течение всего срока действия контракта."),
    ("License: The private partner controls the infrastructure and retains most of the project risks.",
     "Лицензия: частный партнер контролирует инфраструктуру и несет большую часть проектных рисков."),
    ("The concession model is considered the most frequently used model in Europe.",
     "Концессионная модель считается наиболее часто используемой в Европе."),
    ("In the US, startups such as Tesla have been instrumental for the inception of a mature charging infrastructure network.",
     "В США важную роль в создании полноценной сети зарядной инфраструктуры сыграли такие компании, как «Tesla»."),
    ("In Europe, the incumbent network of energy utilities has played a dominant role in the uptake of charging infrastructure.",
     "В Европе доминирующую роль в распространении зарядной инфраструктуры играет сеть действующих энергетических компаний."),
    ("Within Europe, each country has developed a different market model.",
     "В Европе каждая из стран разработала собственную рыночную модель."),
    ("Every service provider that wants to enter the Portuguese market for EV charging will need to register.",
     "Каждый поставщик услуг, желающий выйти на португальский рынок зарядки ЭМ, должен зарегистрироваться."),
    ("Every charging station manufacturer needs to be certified against the requirements.",
     "Каждый изготовитель зарядных станций должен пройти сертификацию на соответствие требованиям."),
    ("Although an open market, there is a strong central coordination to assure standardization.",
     "При всей открытости рынка осуществляется активная центральная координация для обеспечения стандартизации."),
    ("National regulation describes certain technical requirements and protocols such as OCPP.",
     "В национальных правилах описаны определенные технические требования и такие протоколы, как ОПЗС."),
    ("The European Commission provides guidance through the Regulation.",
     "Европейская комиссия выносит рекомендации через Регламент."),
    ("Differences in opinion remain as to what level of harmonization should be prescribed.",
     "Сохраняются разногласия по вопросу о том, какой именно уровень гармонизации следует предписать."),
    ("The type of market — the regulatory framework — differs per region and per country.",
     "Тип рынка — нормативно-правовая база — различается в зависимости от региона и страны."),
    ("Within each market model, both public and private actors are invested in innovations.",
     "В рамках каждой рыночной модели как государственные, так и частные структуры принимают участие в инновациях."),
    ("Openness and interoperability are fundamental aspects of a market-driven sector like electric mobility and charging infrastructure.",
     "Основополагающими критериями такого сектора, как электромобильность и зарядная инфраструктура, являются открытость и интероперабельность."),
    ("Interoperability is a concept that refers to the ability of systems to operate together.",
     "Интероперабельность относится к числу понятий, обозначающих способность систем действовать совместно."),
    ("Reduction of installation and integration costs, mainly due to not requiring conversion services.",
     "Снижение затрат на установку и интеграцию, в основном за счет отсутствия потребности в услугах по преобразованию."),
    ("Efficient scale-up of services through reuse of interoperable components.",
     "Эффективное расширение масштабов оказания услуг путем повторного использования интероперабельных компонентов."),
    ("A better competitive environment as technology 'lock-in' is prevented.",
     "Улучшение конкурентной среды с учетом предотвращения замораживания технологий."),
    ("When systems are interoperable, they may still make use of proprietary tools to achieve interoperability.",
     "Если системы совместимы, то для достижения интероперабельности они могут использовать собственные инструменты."),
    ("The Committee defines six aspects of openness for open international standardization processes.",
     "Комитет определяет шесть аспектов открытости для открытых процессов международной стандартизации."),
    ("In the context of EV charging, an open sector relies on both interoperability and openness.",
     "В контексте зарядки ЭМ открытый сектор опирается как на интероперабельность, так и на открытость."),
    ("Policymakers should periodically assess future developments to ensure a competitive marketplace.",
     "Директивным органам следует периодически оценивать будущие изменения для обеспечения конкурентного рынка."),
    ("The Smart Grid Architecture Model identifies different layers of interoperability.",
     "Модель архитектуры интеллектуальной сети выявляет различные уровни интероперабельности."),
    ("Hardware layer focuses on interoperability of connectors and plugs.",
     "Аппаратный уровень, в контексте которого основное внимание уделяется интероперабельности разъемов и штекеров."),
    ("Communication layer involves seamless communication between systems.",
     "Коммуникационный уровень, в рамках которого предполагается бесперебойная связь между системами."),
    ("Information layer, the information that is being exchanged needs to be recognized.",
     "Информационный уровень, в рамках которого речь идет об информации, которая должна распознаваться."),
    ("Service layer requires standardization to allow for services such as payment and roaming.",
     "Сервисный уровень, требующий стандартизации для обеспечения таких услуг, как платежи и роуминг."),
    ("Business layer necessitates a non-discriminatory regulatory framework to describe the 'rules of the game'.",
     "Бизнес-уровень, требующий недискриминационной нормативной базы для описания «правил игры»."),
    ("The degree of interoperability may vary per market.",
     "Степень интероперабельности может варьироваться в зависимости от рынка."),
    ("The deployment of charging infrastructure has started with operators offering access to their networks.",
     "Развертывание зарядной инфраструктуры началось с того, что операторы стали предлагать доступ к своим сетям."),
    ("Direct payment solutions utilize interoperable payment systems such as credit card terminals.",
     "Для прямых платежей используются такие интероперабельные платежные системы, как терминалы кредитных карт."),
    ("The subscription model offers several advantages, such as a customized customer experience.",
     "Модель подписки имеет некоторые преимущества, такие как индивидуальный подход к клиенту."),
    ("The concept of roaming originates from the telecom sector.",
     "Концепция роуминга происходит из телекоммуникационного сектора."),
    ("Implementing a contract-based model with interoperable EV roaming offers several benefits.",
     "Внедрение основанной на договоре модели с интероперабельным роумингом для ЭМ дает ряд преимуществ."),
    ("EV drivers can charge their vehicles at any charging network without the need for multiple access methods.",
     "Водители могут заряжать свои ЭМ от любой зарядной сети без необходимости использования нескольких методов доступа."),
    ("Distribution System Operators, through CPOs, gain better control over grid congestion.",
     "Операторы распределительных сетей через операторов зарядных станций контролируют перегрузку сети."),
    ("When combining the above insights into a future proof market model, the following picture arises.",
     "Если объединить вышеизложенные соображения в рамках формата будущей рыночной модели, то возникает картина."),
    ("An overview of most widely used charging protocols and standards is described below.",
     "Обзор наиболее широко используемых протоколов и стандартов, касающихся зарядки, приведен ниже."),
    ("Open Charge Point Protocol has been developed to make it possible to connect different types of charging stations.",
     "Открытый протокол зарядной станции был разработан с целью обеспечить возможность подключения зарядных станций различных типов."),
    ("The deployment of charging infrastructure follows a similar pattern to other innovative sectors.",
     "Развертывание зарядной инфраструктуры происходит по той же схеме, что и в других инновационных секторах."),
    ("The focus on user friendliness can be made concrete by defining a customer journey.",
     "Ориентация на удобство для пользователя может быть конкретизирована посредством определения поездки клиента."),
    ("EV drivers are proactive and investing a significant amount of time in gathering information.",
     "Водители электромобилей инициативны и тратят значительное время на сбор информации."),
    ("The primary selection criteria of charging stations are availability, charging speed and costs.",
     "Основными критериями выбора зарядных станций являются доступность, скорость зарядки и стоимость."),
    ("A service benchmark has been developed to measure the user-friendliness of public charging.",
     "Для определения степени удобства использования общественного варианта зарядки был разработан контрольный показатель качества обслуживания клиентов."),
    ("Part of the service definition of a charging station is the reliability.",
     "Составным элементом определения качества обслуживания в рамках зарядной станции является надежность."),
    ("Reliability can be measured by the so-called up-time.",
     "Оценка надежности зарядного устройства проводится с использованием времени безотказной работы."),
]

VARIANT = 10
TMX_FILE = f"variant_{VARIANT}.tmx"
DOMAIN = "Автомобилестроение и электротранспорт"


def build_tmx(pairs: Sequence[Tuple[str, str]], path: str,
              src_lang: str = "en", tgt_lang: str = "ru",
              domain: str = DOMAIN) -> str:
    """Сгенерировать валидную память переводов в формате TMX 1.4b."""
    from datetime import datetime, timezone

    created = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

    tmx = ET.Element("tmx", {"version": "1.4"})
    ET.SubElement(tmx, "header", {
        "creationtool": "lab01-generator",
        "creationtoolversion": "1.0",
        "segtype": "sentence",
        "o-tmf": "TMX",
        "adminlang": "en",
        "srclang": src_lang,
        "datatype": "plaintext",
        "creationdate": created,
    })
    body = ET.SubElement(tmx, "body")

    for index, (source, target) in enumerate(pairs, start=1):
        tu = ET.SubElement(body, "tu", {"tuid": str(index), "creationdate": created})
        ET.SubElement(tu, "prop", {"type": "domain"}).text = domain
        for lang, text in ((src_lang, source), (tgt_lang, target)):
            tuv = ET.SubElement(tu, "tuv", {XML_LANG: lang})
            ET.SubElement(tuv, "seg").text = text

    indent_tree(tmx)
    xml_body = ET.tostring(tmx, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write('<!DOCTYPE tmx SYSTEM "tmx14.dtd">\n')
        handler.write(xml_body)
    return path


build_tmx(PARALLEL_CORPUS, TMX_FILE)
print(f"Создан {TMX_FILE}: {os.path.getsize(TMX_FILE)} байт, "
      f"единиц перевода: {len(PARALLEL_CORPUS)}\n")
print("--- первые 20 строк файла ---")
with open(TMX_FILE, encoding="utf-8") as handler:
    for line in list(handler)[:20]:
        print(line.rstrip())

Создан variant_10.tmx: 56024 байт, единиц перевода: 100

--- первые 20 строк файла ---
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE tmx SYSTEM "tmx14.dtd">
<tmx version="1.4">
  <header creationtool="lab01-generator" creationtoolversion="1.0" segtype="sentence" o-tmf="TMX" adminlang="en" srclang="en" datatype="plaintext" creationdate="20261020T083616Z"/>
  <body>
    <tu tuid="1" creationdate="20261020T083616Z">
      <prop type="domain">Автомобилестроение и электротранспорт</prop>
      <tuv xml:lang="en">
        <seg>In 2022, more than 600,000 public slow charging points were installed globally.</seg>
      </tuv>
      <tuv xml:lang="ru">
        <seg>В 2022 году во всем мире было установлено более 600 000 общественных пунктов медленной зарядки.</seg>
      </tuv>
    </tu>
    <tu tuid="2" creationdate="20261020T083616Z">
      <prop type="domain">Автомобилестроение и электротранспорт</prop>
      <tuv xml:lang="en">
        <seg>The European Alternative Fuels Observatory prov

In [50]:
VARIANT = 10
SURNAME = "Petrova"

TBX_FILE = f"variant_{VARIANT}.tbx"
TMX_FILE = f"variant_{VARIANT}.tmx"

print(f"TBX : {TBX_FILE} ({os.path.getsize(TBX_FILE)} байт)")
print(f"TMX : {TMX_FILE} ({os.path.getsize(TMX_FILE)} байт)")

my_glossary = parse_tbx(TBX_FILE)
print(f"\nГлоссарий: {len(my_glossary)} терминов")

my_tm = parse_tmx(TMX_FILE)
print(f"Память переводов: {len(my_tm)} сегментов")

QUERIES: List[Dict[str, Optional[str]]] = [
    {
        "label": "1. ICE — точное совпадение с совпадением контекста",
        # Берем 7-е предложение из корпуса
        "query": "A charging network with limited interoperability can pose challenges.",
        # Контекст — 6-е предложение (идет прямо перед ним)
        "context": "When establishing specific target values for the aforementioned KPI's, it is essential to consider them.",
    },
    {
        "label": "2. Fuzzy — заменено одно слово",
        # Берем 10-е предложение: "An AC charging station directly provides AC to the EV."
        # Меняем "EV" на "vehicle"
        "query": "An AC charging station directly provides AC to the vehicle.",
        "context": None,
    },
    {
        "label": "3. No match — сегмент вне предметной области",
        "query": "The quarterly financial report was approved by the board.",
        "context": None,
    },
]

all_matches: List[Dict[str, Any]] = []
for case in QUERIES:
    matches = calculate_fuzzy_match(case["query"], my_tm, threshold=75.0,
                                    context_before=case["context"])
    print("=" * 100)
    print(case["label"])
    print(f"Запрос: {case['query']}")
    if not matches:
        print("  -> совпадений выше порога 75 % не найдено, сегмент переводится с нуля")
        continue
    for rank, match in enumerate(matches, start=1):
        print(f"  #{rank} {match['score']:6.2f} %  [{match['match_type']:<12}] tuid={match['tuid']}")
        print(f"      TM src: {match['source_segment']}")
        print(f"      TM tgt: {match['target_segment']}")
        match["query"] = case["query"]
        all_matches.append(match)

matches_df = pd.DataFrame(all_matches)
print("=" * 100)
print(f"\nВсего кандидатов выше порога: {len(matches_df)}")

GLOSSARY_CSV = f"glossary_smartcat_v{VARIANT}.csv"
TM_CSV = f"tm_segments_v{VARIANT}.csv"
FUZZY_CSV = f"fuzzy_report_v{VARIANT}.csv"

smartcat_df = pd.DataFrame({
    "Source (en)": my_glossary["source_term"],
    "Target (ru)": my_glossary["target_term"],
    "Comment": my_glossary["definition"],
})
smartcat_df.to_csv(GLOSSARY_CSV, index=False, encoding="utf-8-sig")
my_tm[["tuid", "source_segment", "target_segment", "domain"]].to_csv(
    TM_CSV, index=False, encoding="utf-8-sig")
matches_df[["query", "tuid", "source_segment", "target_segment",
            "score", "lev_ratio", "token_sort_ratio", "match_type"]].to_csv(
    FUZZY_CSV, index=False, encoding="utf-8-sig")

print("\nЭкспортированные файлы:")
for name in [TBX_FILE, TMX_FILE, GLOSSARY_CSV, TM_CSV, FUZZY_CSV]:
    print(f"  {name:<28} {os.path.getsize(name):>7} байт")

print("\nИтоговая таблица совпадений:")
matches_df[["query", "tuid", "score", "match_type"]]

TBX : variant_10.tbx (73813 байт)
TMX : variant_10.tmx (56024 байт)

Глоссарий: 100 терминов
Память переводов: 100 сегментов
1. ICE — точное совпадение с совпадением контекста
Запрос: A charging network with limited interoperability can pose challenges.
  #1 100.00 %  [ICE (101%)  ] tuid=7
      TM src: A charging network with limited interoperability can pose challenges.
      TM tgt: Зарядная сеть с ограниченной интероперабельностью может создавать проблемы.
2. Fuzzy — заменено одно слово
Запрос: An AC charging station directly provides AC to the vehicle.
  #1  89.66 %  [Fuzzy       ] tuid=10
      TM src: An AC charging station directly provides AC to the EV.
      TM tgt: Станция зарядки переменным током подает переменный ток напрямую в ЭМ.
3. No match — сегмент вне предметной области
Запрос: The quarterly financial report was approved by the board.
  -> совпадений выше порога 75 % не найдено, сегмент переводится с нуля

Всего кандидатов выше порога: 2

Экспортированные файлы:
  va

,query,tuid,score,match_type
0,A charging network with limited interoperability can pos...,7,100.00,ICE (101%)
1,An AC charging station directly provides AC to the vehicle.,10,89.66,Fuzzy


## Блок 7. Выводы

Ответьте письменно (3–5 предложений на пункт):

1. Чем концептуально-ориентированная организация TBX отличается от
   сегментно-ориентированной организации TMX?
В TBX главный элемент — это `termEntry`, то есть отдельное понятие (как «зарядная станция»). Внутри него лежат его переводы на разные языки, часть речи и определения. Поэтому TBX — это справочник терминов. А TMX устроен иначе: его основа — `tu`, то есть готовая пара предложений (оригинал и перевод). TMX нужен, чтобы искать, встречался ли такой текст в прошлом. В лабораторной TBX дал глоссарий, а TMX — базу для поиска совпадений.

2. Почему `xml:lang` требует особой обработки при парсинге и как вы её решили?
При разборе XML-файлов питон не видит `xml:lang` как обычный атрибут `lang`, потому что этот тег принадлежит встроенному пространству имен. Если написать `element.get("lang")`, вернется None. Чтобы это исправить, в коде создана константа `XML_LANG` с полным адресом пространства имен, которая используется в функциях. Тогда английские и русские секции считываются корректно.

3. В каких случаях Levenshtein ratio даёт заниженную оценку сходства и как это компенсирует Token Sort Ratio?
Функция Левенштейна смотрит на символы по порядку. Если в предложениях переставить слова местами, она выдаст мало процентов, хотя по смыслу фразы могут быть почти одинаковыми. Token Sort Ratio перед сравнением сортирует слова в обеих строках по алфавиту. В алгоритме `calculate_fuzzy_match` берется лучший результат из двух (`max(lev, tsr)`), поэтому совпадения находятся даже там, где сильно изменен порядок слов.

4. Чем ICE (101 %) отличается от обычного 100 %-совпадения и почему это важно для стоимости проекта в Smartcat?
Разница кроется в окружении текста. Exact (100 %) фиксирует только то, что строка полностью совпала с базой. ICE (101 %) — это «контекстно-точное» совпадение, при котором гарантируется, что и предыдущая строка в документе совпадает с той, что была в памяти переводов. В коде это реализуется благодаря проверке параметра `context_before`. Для бюджета локализации это критично: статус ICE позволяет применить минимальный тариф (обычно 10–20 % от базовой ставки), так как валидация окружения уже пройдена и переводчику остается лишь подтвердить вставку.

5. Какие проблемы возникли при автоматической подстановке терминов и как вы их решили?
Главная сложность при разметке текста глоссарием — это пересечение терминов. Если алгоритм сначала заменит короткое слово (например, «station»), он разрушит более длинное словосочетание («charging station»). Дополнительно существует риск ложных срабатываний внутри других слов. Для обхода этих ограничений в функции `apply_term_glossary` реализован трехступенчатый подход: приоритет отдается длинным фразам (сортировка по убыванию длины), поиск ведется строго по границам слов с помощью регулярного выражения, а спецсимволы `\x00` используются как временные заглушки для защиты уже размеченных фрагментов от повторного захвата.


**Ваши выводы:**

Лабораторная работа продемонстрировала, как программные методы обработки структурированных данных применяются в лингвистике. На примере предметной области «Автомобилестроение и электротранспорт» была создана база из 100 терминов (TBX) и корпус из 100 предложений (TMX). В ходе тестирования алгоритмов выяснилось, что сочетание символьного и токенного сравнения позволяет гибко настраивать порог срабатывания для поиска нечетких совпадений (Fuzzy Match), а проверка контекста (ICE) дает возможность минимизировать затраты на редактуру. Автоматическая разметка текста по глоссарию с учетом границ слов и приоритета длинных фраз доказала свою эффективность для соблюдения единообразия узкоспециализированных терминов.


_______________________________________________

---

### Что сдать на проверку

1. Ссылка на Colab-ноутбук `lab_01_Фамилия.ipynb` (доступ «для всех по ссылке»).
2. Файлы `variant_NN.tbx`, `variant_NN.tmx`, `glossary_smartcat.csv`.
3. Отчёт с выводами и скриншотом импортированного глоссария в Smartcat.